# Cybersecurity Incident Analyzer — Fase 1


---
## 1. Instalação de Dependências e Imports

Instalamos as bibliotecas necessárias para carregar modelos pré-treinados, manipular o dataset e avaliar os resultados.


In [1]:
!pip install -q transformers torch pandas numpy sentencepiece accelerate pydantic


In [2]:
import json
import re
import textwrap
from enum import Enum

import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel, Field, field_validator
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline,
)

device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"Dispositivo em uso: {device_name}")

if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")


Dispositivo em uso: GPU
GPU detectada: Tesla T4


In [3]:
def print_wrapped(text, width=100):
    for line in str(text).splitlines() or [""]:
        if line.strip():
            print(
                textwrap.fill(
                    line,
                    width=width,
                    break_long_words=False,
                    break_on_hyphens=False,
                )
            )
        else:
            print()


---
## 2. Dataset de Incidentes (Exemplos Fictícios)

O dataset abaixo contém **10 incidentes fictícios**. Cada exemplo inclui:

- o relato do incidente em inglês;
- comentários e explicações em português.

Nesta fase, os textos são usados para demonstrar sumarização e extração direta de entidades com NER.


In [4]:
incidents = [
    {
        "id": 1,
        "text": "On June 14th, multiple employees in the Finance department reported receiving an email claiming to be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract penalties'. One employee clicked the link and entered their corporate credentials on a fake Office 365 login page before realizing the site was illegitimate. IT Security was notified and the employee's account was immediately disabled pending investigation.",
        "expected_extraction": {
            "incident_type": "Phishing",
            "affected_asset": "Finance employee account",
            "observables": [
                "cfo-finance-corp.com",
                "Office 365"
            ],
            "impact": "An employee submitted corporate credentials to a fake Office 365 login page.",
            "response_actions": [
                "IT Security was notified",
                "The employee account was disabled pending investigation"
            ]
        }
    },
    {
        "id": 2,
        "text": "At 03:12 AM, the file server hosting the HR department's shared drive became unresponsive. Upon investigation, system administrators discovered that thousands of files had been encrypted and renamed with a '.lockedcorp' extension. A ransom note demanding 5 Bitcoin was found in every affected directory, threatening to leak employee personal data if payment was not made within 72 hours. Backups from the previous night appear unaffected and are being verified for integrity.",
        "expected_extraction": {
            "incident_type": "Ransomware",
            "affected_asset": "HR shared file server",
            "observables": [
                ".lockedcorp",
                "5 Bitcoin",
                "72 hours"
            ],
            "impact": "Thousands of files on the HR shared file server were encrypted.",
            "response_actions": [
                "Backups are being verified for integrity"
            ]
        }
    },
    {
        "id": 3,
        "text": "Endpoint detection software flagged unusual behavior on a marketing department laptop: a previously unseen executable was making repeated outbound connections to an IP address associated with known command-and-control infrastructure. The process was attempting to enumerate browser-stored credentials and capture keystrokes. The laptop was isolated from the network and a forensic image was taken for analysis.",
        "expected_extraction": {
            "incident_type": "Malware",
            "affected_asset": "Marketing laptop",
            "observables": [
                "command-and-control infrastructure",
                "browser-stored credentials",
                "keystrokes"
            ],
            "impact": "A malicious executable attempted to steal credentials and capture keystrokes from a laptop.",
            "response_actions": [
                "The laptop was isolated from the network",
                "A forensic image was taken for analysis"
            ]
        }
    },
    {
        "id": 4,
        "text": "Security monitoring detected that an employee's VPN credentials were used to log in simultaneously from two geographically distant locations — one from the corporate office in Sao Paulo and another from an IP address registered in Eastern Europe, less than ten minutes apart. The employee confirmed they had not traveled and had not shared their password. A password reset and credential rotation was initiated immediately, along with a review of recently accessed systems.",
        "expected_extraction": {
            "incident_type": "Credential Theft",
            "affected_asset": "Employee VPN account",
            "observables": [
                "Sao Paulo",
                "Eastern Europe"
            ],
            "impact": "The employee VPN account was accessed from impossible-travel locations within minutes.",
            "response_actions": [
                "A password reset was initiated",
                "Credential rotation was initiated",
                "Recently accessed systems were reviewed"
            ]
        }
    },
    {
        "id": 5,
        "text": "A system administrator with privileged access to the customer database was found to have exported a large volume of customer records to a personal USB drive shortly after receiving a negative performance review. The export occurred outside of normal working hours and was not associated with any approved business task. Data Loss Prevention (DLP) tooling flagged the transfer for review.",
        "expected_extraction": {
            "incident_type": "Insider Threat",
            "affected_asset": "Customer database",
            "observables": [
                "personal USB drive",
                "Data Loss Prevention"
            ],
            "impact": "A privileged administrator exported customer records to a personal USB drive.",
            "response_actions": [
                "The transfer was flagged for review by DLP tooling"
            ]
        }
    },
    {
        "id": 6,
        "text": "The authentication logs for the company's customer-facing web portal show over 40,000 failed login attempts within a two-hour window, originating from a distributed set of IP addresses and targeting a list of common username patterns. The pattern is consistent with an automated credential-stuffing or brute-force attack attempting to compromise customer accounts using leaked password lists.",
        "expected_extraction": {
            "incident_type": "Brute Force Attack",
            "affected_asset": "Customer-facing web portal",
            "observables": [
                "40,000 failed login attempts",
                "distributed IP addresses",
                "leaked password lists"
            ],
            "impact": "The customer-facing web portal was targeted by a large credential-stuffing campaign.",
            "response_actions": []
        }
    },
    {
        "id": 7,
        "text": "The company's primary e-commerce website became completely unreachable for approximately 45 minutes during a major promotional sale. Network monitoring showed an abnormal surge of incoming traffic, estimated at over 200 times normal volume, originating from thousands of distinct IP addresses worldwide and overwhelming the load balancers. The traffic pattern strongly suggests a coordinated denial-of-service attack timed to coincide with peak sales traffic.",
        "expected_extraction": {
            "incident_type": "DDoS",
            "affected_asset": "Primary e-commerce website",
            "observables": [
                "45 minutes",
                "200 times normal volume",
                "thousands of distinct IP addresses"
            ],
            "impact": "The primary e-commerce website became unreachable during a traffic flood.",
            "response_actions": []
        }
    },
    {
        "id": 8,
        "text": "A help desk technician received a phone call from someone claiming to be a senior executive who had 'forgotten' their password and urgently needed it reset to access an important client presentation. The caller was able to provide the executive's employee ID and date of birth, likely gathered from a previous data breach, and pressured the technician to bypass standard identity verification procedures.",
        "expected_extraction": {
            "incident_type": "Phishing",
            "affected_asset": "Executive account",
            "observables": [
                "employee ID",
                "date of birth",
                "phone call"
            ],
            "impact": "A caller used social engineering to pressure the help desk into resetting an executive account password.",
            "response_actions": []
        }
    },
    {
        "id": 9,
        "text": "Anti-virus software on a developer's workstation quarantined a file disguised as a legitimate npm package dependency that had been recently added to a project. Analysis revealed the package contained obfuscated code designed to harvest environment variables, including API keys and cloud credentials, and transmit them to an external server upon installation.",
        "expected_extraction": {
            "incident_type": "Malware",
            "affected_asset": "Developer workstation",
            "observables": [
                "npm package dependency",
                "API keys",
                "cloud credentials"
            ],
            "impact": "A malicious package attempted to harvest API keys and cloud credentials from a developer workstation.",
            "response_actions": [
                "Anti-virus software quarantined the file"
            ]
        }
    },
    {
        "id": 10,
        "text": "Several customer support agents reported that their session tokens appeared to be reused from unfamiliar devices shortly after they had logged into the support ticketing platform from a coffee shop's public Wi-Fi network. Investigation suggests the sessions may have been intercepted via an unsecured network, allowing an attacker to hijack active sessions without needing the original password.",
        "expected_extraction": {
            "incident_type": "Credential Theft",
            "affected_asset": "Support ticketing platform sessions",
            "observables": [
                "session tokens",
                "public Wi-Fi",
                "unfamiliar devices"
            ],
            "impact": "Active support platform sessions may have been hijacked after token interception on public Wi-Fi.",
            "response_actions": []
        }
    }
]

df_incidents = pd.DataFrame(
    {
        "id": [incident["id"] for incident in incidents],
        "incident_type": [incident["expected_extraction"]["incident_type"] for incident in incidents],
        "affected_asset": [incident["expected_extraction"]["affected_asset"] for incident in incidents],
    }
)

print(f"Total de incidentes no dataset: {len(df_incidents)}")
df_incidents


Total de incidentes no dataset: 10


,id,incident_type,affected_asset
0,1,Phishing,Finance employee account
1,2,Ransomware,HR shared file server
2,3,Malware,Marketing laptop
3,4,Credential Theft,Employee VPN account
4,5,Insider Threat,Customer database
5,6,Brute Force Attack,Customer-facing web portal
6,7,DDoS,Primary e-commerce website
7,8,Phishing,Executive account
8,9,Malware,Developer workstation
9,10,Credential Theft,Support ticketing platform sessions


---
## 3. Tarefa 1 — Sumarização de Incidentes

Relatos de incidentes em ambientes reais frequentemente chegam em formato longo e narrativo. A sumarização automática ajuda o analista a entender o núcleo do caso sem ler o texto inteiro antes da triagem.

Utilizamos o modelo **`facebook/bart-large-cnn`**, um modelo encoder-decoder pré-treinado para tarefas de sumarização em inglês.


In [5]:
summarization_model_name = "facebook/bart-large-cnn"
summarization_tokenizer = AutoTokenizer.from_pretrained(summarization_model_name)
summarization_model = AutoModelForSeq2SeqLM.from_pretrained(summarization_model_name)
summarization_model.eval()

if torch.cuda.is_available():
    summarization_model = summarization_model.to("cuda")


def summarize_text(text, max_length=60, min_length=15):
    inputs = summarization_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    model_device = next(summarization_model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        summary_ids = summarization_model.generate(
            **inputs,
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            early_stopping=True,
        )

    return summarization_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [6]:
example_incident = incidents[0]
summary = summarize_text(example_incident["text"])

print("INCIDENTE ORIGINAL:")
print_wrapped(example_incident["text"])
print()
print("SUMARIZACAO GERADA:")
print_wrapped(summary)


INCIDENTE ORIGINAL:
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a
spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract
penalties'. One employee clicked the link and entered their corporate credentials on a fake Office
365 login page before realizing the site was illegitimate. IT Security was notified and the
employee's account was immediately disabled pending investigation.

SUMARIZACAO GERADA:
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO. The email contained a spoofed domain (cfo-finance-corp.com) and urged recipients to
act within one hour to avoid 'contract penalties' One


In [7]:
summaries = []
for incident in incidents:
    summaries.append(
        summarize_text(
            incident["text"],
            max_length=50,
            min_length=10,
        )
    )

summary_df = pd.DataFrame(
    {
        "id": [incident["id"] for incident in incidents],
        "incident_type": [incident["expected_extraction"]["incident_type"] for incident in incidents],
        "summary": summaries,
    }
)

summary_df


,id,incident_type,summary
0,1,Phishing,"On June 14th, multiple employees in the Financ..."
1,2,Ransomware,Thousands of files were encrypted and renamed ...
2,3,Malware,A marketing department laptop was targeted by ...
3,4,Credential Theft,Security monitoring detected that an employee'...
4,5,Insider Threat,A system administrator exported a large volume...
5,6,Brute Force Attack,The authentication logs for the company's cust...
6,7,DDoS,The company's primary e-commerce website becam...
7,8,Phishing,A help desk technician received a phone call f...
8,9,Malware,Anti-virus software quarantined a file disguis...
9,10,Credential Theft,Several customer support agents reported that ...


---
## 4. Tarefa 2 — Extração com NER

Nesta etapa, usamos um modelo **NER pré-treinado** para identificar entidades diretamente no texto do incidente.

Aqui a saída é mantida de forma simples: mostramos **o retorno do próprio pipeline de NER**, sem regras adicionais para montar campos estruturados.


In [8]:
ner_model_name = "dslim/bert-base-NER"
ner_extractor = pipeline(
    "token-classification",
    model=ner_model_name,
    aggregation_strategy="simple",
    device=device,
)


def extract_named_entities(text):
    return ner_extractor(text)


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
for incident_id in [1, 4, 9]:
    incident = next(item for item in incidents if item["id"] == incident_id)
    extracted_entities = extract_named_entities(incident["text"])

    print(f"INCIDENTE #{incident_id}")
    print_wrapped(incident["text"])
    print()
    print("SAIDA DO NER:")
    for entity in extracted_entities:
        print(entity)
    print("-" * 80)


INCIDENTE #1
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a
spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract
penalties'. One employee clicked the link and entered their corporate credentials on a fake Office
365 login page before realizing the site was illegitimate. IT Security was notified and the
employee's account was immediately disabled pending investigation.

SAIDA DO NER:
{'entity_group': 'ORG', 'score': np.float32(0.86613643), 'word': 'Finance department', 'start': 40, 'end': 58}
{'entity_group': 'ORG', 'score': np.float32(0.7444848), 'word': 'CF', 'start': 111, 'end': 113}
{'entity_group': 'MISC', 'score': np.float32(0.9689775), 'word': 'Office 365', 'start': 389, 'end': 399}
{'entity_group': 'ORG', 'score': np.float32(0.99692655), 'word': 'IT Security', 'start': 455, 'end': 466}

In [10]:
ner_results = []
for incident in incidents:
    ner_results.append(
        {
            "id": incident["id"],
            "ner_entities": extract_named_entities(incident["text"]),
        }
    )

ner_results_df = pd.DataFrame(ner_results)
ner_results_df


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,ner_entities
0,1,"[{'entity_group': 'ORG', 'score': 0.86613643, ..."
1,2,"[{'entity_group': 'ORG', 'score': 0.51365846, ..."
2,3,[]
3,4,"[{'entity_group': 'ORG', 'score': 0.45450038, ..."
4,5,"[{'entity_group': 'MISC', 'score': 0.94700605,..."
5,6,[]
6,7,[]
7,8,[]
8,9,[]
9,10,"[{'entity_group': 'MISC', 'score': 0.97701454,..."


---
## 5. Leitura da Saída do NER

Cada item retornado pelo pipeline contém, em geral:

- `entity_group`: o tipo agregado da entidade reconhecida;
- `score`: a confiança do modelo;
- `word`: o trecho textual identificado;
- `start` e `end`: a posição da entidade no texto original.

Esse formato já é suficiente para inspecionar rapidamente quais pessoas, organizações, localidades e outros trechos relevantes o modelo reconheceu em cada relato.


In [11]:
entity_group_counts = {}

for row in ner_results:
    for entity in row["ner_entities"]:
        entity_group = entity["entity_group"]
        entity_group_counts[entity_group] = entity_group_counts.get(entity_group, 0) + 1

entity_group_df = pd.DataFrame(
    {
        "entity_group": list(entity_group_counts.keys()),
        "count": list(entity_group_counts.values()),
    }
).sort_values("count", ascending=False)

entity_group_df


,entity_group,count
1,MISC,8
0,ORG,6
2,LOC,2


---
## 6. Tokenização e Arquiteturas Relevantes

### Encoder-decoder

Modelos encoder-decoder são apropriados para tarefas de **transformação de sequência**, em que uma entrada textual precisa ser convertida em outra saída textual mais compacta ou estruturada. Neste notebook, usamos essa família para a **sumarização** com `facebook/bart-large-cnn`.

### Token classification / encoder-only

Modelos de token classification rotulam trechos do texto diretamente na entrada. Isso é útil para **Named Entity Recognition (NER)** e extração direta de entidades. Neste notebook, usamos essa família com `dslim/bert-base-NER`.

### Resumo comparativo

| Componente | Tipo de modelo | Papel na Fase 1 |
|---|---|---|
| `facebook/bart-large-cnn` | Encoder-decoder | Sumarizar relatos de incidentes |
| `dslim/bert-base-NER` | Token classification | Retornar entidades reconhecidas no texto |


---
## 7. Limitações Observadas Nesta Fase

Mesmo com o escopo mais enxuto desta fase, ainda existem limitações importantes.

### Cobertura de entidades

Um modelo NER genérico não foi treinado especificamente para o domínio de incidentes de segurança. Por isso, vários observáveis relevantes, como extensões maliciosas, nomes de técnicas ou artefatos específicos, podem não ser reconhecidos como entidades.

### Saída bruta do modelo

A saída do pipeline mostra somente as entidades detectadas pelo modelo, com seus rótulos e scores. Ela não organiza automaticamente essas entidades em um schema operacional de incidente.

### Conhecimento pré-treinado estático

Todos os modelos usados aqui dependem apenas do conhecimento aprendido no pré-treinamento. Eles não conhecem:

- playbooks internos da organização;
- inventário real de ativos críticos;
- incidentes passados da empresa;
- ameaças e indicadores publicados após o corte de treinamento.


# Cybersecurity Incident Analyzer — Fase 2

## Prompt Engineering e Saídas Controladas

Este bloco continua o mesmo dataset da Fase 1 e aplica três técnicas de prompt engineering à triagem automatizada de incidentes de cibersegurança para times SOC.

| # | Técnica | Ideia central |
|---|---------|---------------|
| 1 | **Role + Context + Task + Output Format (RCTOF)** | Prompt totalmente especificado com role priming e esquema JSON explícito |
| 2 | **Guided Chain of Thought (GCoT)** | O prompt guia um raciocínio em etapas antes da saída final |
| 3 | **Meta Prompting** | O LLM gera o prompt ideal; esse prompt conduz a análise final |

As três técnicas usam o **mesmo relato de incidente** e miram o **mesmo esquema de saída em JSON**.


---
## 1. Setup e Imports

In [12]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 768

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
elif torch.cuda.is_available():
    MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MODEL_DEVICE = next(model.parameters()).device

print(f"Model  : {MODEL_ID}")
print(f"Device : {MODEL_DEVICE}")
print(f"DType  : {MODEL_DTYPE}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model  : Qwen/Qwen2.5-1.5B-Instruct
Device : cuda:0
DType  : torch.bfloat16


---
## 2. Definição do Incidente

Os incidentes são reutilizados diretamente do array `incidents` definido no início deste notebook. Um item é selecionado com `INCIDENT_INDEX` e reaproveitado nas três técnicas para permitir comparação direta.

> **Dataset base:** os 10 incidentes fictícios usados em todo o projeto. A seleção inicial abaixo usa o incidente de índice `0`.


In [13]:
INCIDENT_INDEX = 0
selected_incident = incidents[INCIDENT_INDEX]
INCIDENT_TEXT = selected_incident["text"]

print(f"Loaded {len(incidents)} incidents from the in-memory incidents array")
print(f"Using incident #{selected_incident['id']}")
print(f"Incident length: {len(INCIDENT_TEXT)} characters")
print()
print("─" * 70)
print_wrapped(INCIDENT_TEXT, width=70)
print("─" * 70)


Loaded 10 incidents from the in-memory incidents array
Using incident #1
Incident length: 554 characters

──────────────────────────────────────────────────────────────────────
On June 14th, multiple employees in the Finance department reported
receiving an email claiming to be from the CFO, requesting urgent wire
transfers to a new vendor account. The email contained a spoofed
domain (cfo-finance-corp.com) and urged recipients to act within one
hour to avoid 'contract penalties'. One employee clicked the link and
entered their corporate credentials on a fake Office 365 login page
before realizing the site was illegitimate. IT Security was notified
and the employee's account was immediately disabled pending
investigation.
──────────────────────────────────────────────────────────────────────


---
## 3. Esquema de Saída (Pydantic v2)

As três técnicas precisam produzir saídas compatíveis com este esquema fixo.

O Pydantic v2 valida:
- `severity` pertence a `low | medium | high | critical` (Enum, rejeitando strings inválidas)
- `confidence` é um float `0.0 ≤ x ≤ 1.0`
- `recommended_actions` é uma lista não vazia de strings não vazias
- `summary` tem no mínimo 10 caracteres

In [34]:
class SeverityLevel(str, Enum):
    low = "low"
    medium = "medium"
    high = "high"
    critical = "critical"


class IncidentAnalysis(BaseModel):
    incident_type: str = Field(..., description="Type of cybersecurity incident")
    severity: SeverityLevel = Field(..., description="Severity: low/medium/high/critical")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence score 0.0-1.0")
    summary: str = Field(..., min_length=10, description="2-3 sentence incident summary")
    recommended_actions: list[str] = Field(
        ..., min_length=1, description="Prioritized SOC response actions"
    )

    @field_validator("severity", mode="before")
    @classmethod
    def normalize_severity(cls, v):
        return v.strip().lower() if isinstance(v, str) else v

    @field_validator("confidence")
    @classmethod
    def round_confidence(cls, v: float) -> float:
        return round(v, 3)

    @field_validator("recommended_actions")
    @classmethod
    def actions_not_empty(cls, v: list[str]) -> list[str]:
        if not all(isinstance(a, str) and a.strip() for a in v):
            raise ValueError("All actions must be non-empty strings")
        return v


print("Schema: IncidentAnalysis")
print()
schema = IncidentAnalysis.model_json_schema()
print(json.dumps(schema, indent=2))

Schema: IncidentAnalysis

{
  "$defs": {
    "SeverityLevel": {
      "enum": [
        "low",
        "medium",
        "high",
        "critical"
      ],
      "title": "SeverityLevel",
      "type": "string"
    }
  },
  "properties": {
    "incident_type": {
      "description": "Type of cybersecurity incident",
      "title": "Incident Type",
      "type": "string"
    },
    "severity": {
      "$ref": "#/$defs/SeverityLevel",
      "description": "Severity: low/medium/high/critical"
    },
    "confidence": {
      "description": "Confidence score 0.0-1.0",
      "maximum": 1.0,
      "minimum": 0.0,
      "title": "Confidence",
      "type": "number"
    },
    "summary": {
      "description": "2-3 sentence incident summary",
      "minLength": 10,
      "title": "Summary",
      "type": "string"
    },
    "recommended_actions": {
      "description": "Prioritized SOC response actions",
      "items": {
        "type": "string"
      },
      "minItems": 1,
      "title": "R

---
## 4. Utilitários de Geração, Parsing e Validação

Pipeline completo: `prompt de chat → geração com modelo open source → parse de JSON → validação com Pydantic`

### Tratamento de erros

| Modo de falha | Tratamento |
|---|---|
| JSON envolto em markdown fences | Remoção via regex |
| `json.loads()` falha | `ValueError` com contexto |
| Validação do Pydantic falha | `ValidationError` com detalhe por campo |

In [39]:
def generate_response_text(system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS):
    # Monta a conversa no formato esperado pelo modelo e gera apenas a continuação.
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt_text, return_tensors="pt")
    model_inputs = {k: v.to(MODEL_DEVICE) for k, v in model_inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_tokens = model_inputs["input_ids"].shape[-1]
    completion_tokens = generated_ids.shape[-1] - prompt_tokens
    completion_ids = generated_ids[0][prompt_tokens:]
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()

    return completion_text, prompt_tokens, completion_tokens


def parse_json_response(text):
    # Extrai e interpreta um objeto JSON a partir da saída textual bruta.
    text = text.strip()
    decoder = json.JSONDecoder()

    # 1. Tentativa direta
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. Remoção de cercas de markdown
    for pattern in [r"```json\s*(.*?)\s*```", r"```\s*(.*?)\s*```"]:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                continue

    # 3. Busca incremental pelo primeiro objeto JSON bem-formado.
    for start, char in enumerate(text):
        if char != "{":
            continue
        try:
            candidate, _ = decoder.raw_decode(text[start:])
            if isinstance(candidate, dict):
                return candidate
        except json.JSONDecodeError:
            continue

    raise ValueError(f"No valid JSON found in response:\n{text[:500]}")


def validate_analysis(data):
    # Valida o dicionário contra o schema IncidentAnalysis.
    if isinstance(data, dict) and isinstance(data.get("severity"), str):
        data = {**data, "severity": data["severity"].strip().lower()}
    return IncidentAnalysis(**data)


def repair_analysis_output(raw_text, max_new_tokens=384):
    # Reescreve uma resposta textual livre em JSON estrito quando o parse inicial falha.
    repair_system = (
        "You convert cybersecurity incident analyses into strict JSON. "
        "Return only a valid JSON object with no markdown, headings, or commentary."
    )
    repair_user = (
        "Convert the analysis below into a single valid JSON object.\n\n"
        "Rules:\n"
        "- Return ONLY JSON.\n"
        "- Use exactly this schema:\n"
        + _JSON_SCHEMA
        + "\n\n"
        "- severity must be one of: low, medium, high, critical (lowercase).\n"
        "- confidence must be a float between 0.0 and 1.0.\n"
        "- recommended_actions must be a non-empty list of concise strings.\n"
        "- If the source text is ambiguous, choose the most likely single incident_type and lower confidence instead of inventing facts.\n\n"
        "Source analysis:\n"
        + raw_text[:4000]
    )
    return generate_response_text(repair_system, repair_user, max_new_tokens=max_new_tokens)


def run_analysis(system_prompt, user_prompt, label="Analysis", max_new_tokens=MAX_NEW_TOKENS):
    # Executa a geração, interpreta o JSON e valida a resposta final.
    print(f"[{label}] Sending request...")

    text, input_tokens, output_tokens = generate_response_text(
        system_prompt,
        user_prompt,
        max_new_tokens=max_new_tokens,
    )

    raw_data = None
    analysis = None
    parse_ok = False
    validation_ok = False
    error = None
    repair_attempted = False
    repair_succeeded = False
    repair_text = None
    repair_input_tokens = 0
    repair_output_tokens = 0

    try:
        raw_data = parse_json_response(text)
        parse_ok = True
    except ValueError as e:
        error = f"ParseError: {e}"

    if parse_ok:
        try:
            analysis = validate_analysis(raw_data)
            validation_ok = True
        except Exception as e:
            error = f"ValidationError: {e}"

    if not validation_ok and text:
        print(f"[{label}] Attempting JSON repair...")
        repair_attempted = True
        repair_text, repair_input_tokens, repair_output_tokens = repair_analysis_output(text)
        try:
            raw_data = parse_json_response(repair_text)
            parse_ok = True
            analysis = validate_analysis(raw_data)
            validation_ok = True
            error = None
            repair_succeeded = True
        except Exception as e:
            error = f"RepairFailed: {e}"

    status = "OK" if validation_ok else f"FAILED — {error}"
    print_wrapped(f"[{label}] Status  : {status}")
    print(f"[{label}] Tokens  : {input_tokens} in / {output_tokens} out")
    if repair_attempted:
        print(f"[{label}] Repair  : {repair_input_tokens} in / {repair_output_tokens} out")

    return {
        "label": label,
        "text": text,
        "repair_text": repair_text,
        "raw_data": raw_data,
        "analysis": analysis,
        "parse_ok": parse_ok,
        "validation_ok": validation_ok,
        "error": error,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "repair_attempted": repair_attempted,
        "repair_succeeded": repair_succeeded,
        "repair_input_tokens": repair_input_tokens,
        "repair_output_tokens": repair_output_tokens,
    }


def display_result(result):
    # Exibe de forma legível uma resposta validada.
    sep = "=" * 65
    print(sep)
    print(f"  {result['label']}")
    print(sep)

    if not result["validation_ok"]:
        print_wrapped(f"  ERROR: {result['error']}")
        if result["text"]:
            print("\n  Raw text (first 800 chars):")
            print_wrapped(result['text'][:800])
        if result.get("repair_text"):
            print("\n  Repair text (first 800 chars):")
            print_wrapped(result["repair_text"][:800])
        return

    analysis = result["analysis"]
    print(f"  Incident Type  : {analysis.incident_type}")
    print(f"  Severity       : {analysis.severity.value.upper()}")
    print(f"  Confidence     : {analysis.confidence:.3f}")
    print_wrapped(f"  Summary        : {analysis.summary}")
    print(f"  Actions ({len(analysis.recommended_actions)}):")
    for i, action in enumerate(analysis.recommended_actions, 1):
        print(f"    {i}. {action}")
    if result.get("repair_succeeded"):
        print("  Repair Used    : YES")
    print(f"  Tokens         : {result['input_tokens']} in / {result['output_tokens']} out")


print("Utilities loaded.")

Utilities loaded.


---
## 5. Técnica 1 — Role + Context + Task + Output Format (RCTOF)

### Racional de desenho

O padrão **RCTOF** organiza o prompt em quatro elementos explícitos:

| Elemento | Conteúdo | Propósito |
|----------|----------|------------|
| **Role** | Analista sênior de cibersegurança | Priming de domínio, alinhando o comportamento do modelo |
| **Context** | Cenário de triagem SOC | Enquadramento situacional, reduzindo ambiguidades |
| **Task** | Classificar, avaliar, resumir e recomendar | Escopo explícito da instrução |
| **Output Format** | Esquema JSON exato dentro do prompt | Restrição estrutural da saída |

### Comportamento esperado
O modelo produz JSON diretamente, sem explicações extras. Há apenas uma geração, com baixo overhead.

In [36]:
T1_SYSTEM = (
    "You are a senior cybersecurity analyst specializing in incident response and threat intelligence. "
    "You have extensive experience with SOC operations, incident classification, and triage procedures "
    "across large enterprise environments."
)

_JSON_SCHEMA = (
    '{\n'
    '  "incident_type": "<string: attack category>",\n'
    '  "severity": "<one of: low | medium | high | critical>",\n'
    '  "confidence": <float: 0.0 to 1.0>,\n'
    '  "summary": "<string: 2-3 sentence description of what happened>",\n'
    '  "recommended_actions": [\n'
    '    "<string: first prioritized action>",\n'
    '    "<string: second prioritized action>"\n'
    '  ]\n'
    '}'
)

T1_USER = (
    "## Context\n"
    "Your organization's SOC has received a cybersecurity incident report requiring immediate triage.\n\n"
    "## Task\n"
    "Analyze the incident report below. Identify the attack type, assess severity, estimate your "
    "confidence level, write a concise summary, and list prioritized response actions.\n\n"
    "## Incident Report\n"
    + INCIDENT_TEXT
    + "\n\n"
    "## Output Format\n"
    "Respond with ONLY a valid JSON object — no explanation, no markdown fences, no preamble.\n"
    "Use exactly this schema:\n"
    + _JSON_SCHEMA
)

print(f"System prompt : {len(T1_SYSTEM)} chars")
print(f"User prompt   : {len(T1_USER)} chars")
print()
print("─" * 65)
print_wrapped(T1_USER[:650], width=65)
print("... [truncated]")

System prompt : 233 chars
User prompt   : 1327 chars

─────────────────────────────────────────────────────────────────
## Context
Your organization's SOC has received a cybersecurity incident
report requiring immediate triage.

## Task
Analyze the incident report below. Identify the attack type,
assess severity, estimate your confidence level, write a concise
summary, and list prioritized response actions.

## Incident Report
On June 14th, multiple employees in the Finance department
reported receiving an email claiming to be from the CFO,
requesting urgent wire transfers to a new vendor account. The
email contained a spoofed domain (cfo-finance-corp.com) and urged
recipients to act within one hour to avoid 'contract penalties'.
One employee clicked the link
... [truncated]


In [25]:
result_t1 = run_analysis(T1_SYSTEM, T1_USER, label="T1: RCTOF")
print()
display_result(result_t1)

[T1: RCTOF] Sending request...
[T1: RCTOF] Status  : OK
[T1: RCTOF] Tokens  : 327 in / 94 out

  T1: RCTOF
  Incident Type  : Phishing
  Severity       : HIGH
  Confidence     : 0.900
  Summary        : Employees in the Finance department were tricked into clicking a phishing email that requested urgent wire transfers to a fake vendor account.
  Actions (2):
    1. Enable multi-factor authentication for all financial transactions.
    2. Improve training programs to educate employees about common phishing tactics.
  Tokens         : 327 in / 94 out


---
## 6. Técnica 2 — Guided Chain of Thought (GCoT)

### Racional de desenho

**Chain of Thought** orienta o modelo a organizar o raciocínio em etapas intermediárias antes de chegar à conclusão. Aqui, essas etapas são **explicitamente guiadas** e cada uma aponta para um campo específico da saída.

| Etapa | Alvo do raciocínio | Campo de saída |
|------|---------------------|----------------|
| 1 | Identificação do vetor de ataque | ajuda `incident_type` |
| 2 | Classificação do incidente | `incident_type` |
| 3 | Avaliação de severidade (tríade CIA) | `severity` |
| 4 | Calibração de confiança baseada em evidências | `confidence` |
| 5 | Resumo do incidente | `summary` |
| 6 | Ações priorizadas de resposta | `recommended_actions` |

### Comportamento esperado
Mesmo sem expor o raciocínio intermediário, a estrutura guiada tende a melhorar a calibração e a consistência da saída, ao custo de prompts mais longos.

In [26]:
T2_SYSTEM = (
    "You are a senior cybersecurity analyst with deep expertise in threat analysis and incident response. "
    "You approach every incident with systematic, evidence-based reasoning — examining attack vectors, "
    "blast radius, and response urgency before reaching conclusions."
)

_COT_STEPS = (
    "**Step 1 — Identify the attack vector:**\n"
    "What mechanism was used? Look for: delivery method, exploitation technique, initial access path.\n\n"
    "**Step 2 — Classify the incident type:**\n"
    "Based on the vector and observed behavior, assign an incident type. Consider: Ransomware, "
    "Phishing, Malware, DDoS, Insider Threat, Credential Theft, Brute Force, Supply Chain, or other.\n\n"
    "**Step 3 — Assess severity (low / medium / high / critical):**\n"
    "Evaluate business impact across:\n"
    "- Confidentiality: Was sensitive data exposed or at risk?\n"
    "- Integrity: Were systems or data modified?\n"
    "- Availability: Were services disrupted?\n"
    "- Scope: How many users/systems affected?\n"
    "- Urgency: Is the threat ongoing?\n\n"
    "**Step 4 — Estimate confidence (0.0–1.0):**\n"
    "How certain are you given available evidence?\n"
    "- High (>0.85): clear indicators, multiple corroborating signals\n"
    "- Medium (0.65–0.85): some ambiguity or missing evidence\n"
    "- Low (<0.65): significant uncertainty or contradictory signals\n\n"
    "**Step 5 — Write a concise incident summary (2-3 sentences):**\n"
    "Cover: what happened, who/what was affected, key indicators observed.\n\n"
    "**Step 6 — List prioritized response actions:**\n"
    "Specific, actionable steps for the SOC team, ordered by priority.\n\n"
)

_FINAL_JSON = (
    '{\n'
    '  "incident_type": "<string>",\n'
    '  "severity": "<low|medium|high|critical>",\n'
    '  "confidence": <float 0.0-1.0>,\n'
    '  "summary": "<string>",\n'
    '  "recommended_actions": ["<string>", ...]\n'
    '}'
)

T2_USER = (
    "Analyze this cybersecurity incident using the structured reasoning steps below.\n"
    "Work through each step carefully before producing the final JSON output.\n\n"
    "**Incident Report:**\n"
    + INCIDENT_TEXT
    + "\n\n---\n\n"
    + _COT_STEPS
    + "---\n\n"
    "After completing all steps, output ONLY a valid JSON object:\n"
    + _FINAL_JSON
)

print(f"System prompt : {len(T2_SYSTEM)} chars")
print(f"User prompt   : {len(T2_USER)} chars")
print()
print("─" * 65)
print_wrapped(T2_USER[:650], width=65)
print("... [truncated]")

System prompt : 262 chars
User prompt   : 2190 chars

─────────────────────────────────────────────────────────────────
Analyze this cybersecurity incident using the structured
reasoning steps below.
Work through each step carefully before producing the final JSON
output.

**Incident Report:**
On June 14th, multiple employees in the Finance department
reported receiving an email claiming to be from the CFO,
requesting urgent wire transfers to a new vendor account. The
email contained a spoofed domain (cfo-finance-corp.com) and urged
recipients to act within one hour to avoid 'contract penalties'.
One employee clicked the link and entered their corporate
credentials on a fake Office 365 login page before realizing the
site was illegitimate. IT Security was noti
... [truncated]


In [37]:
#FIXME
result_t2 = run_analysis(T2_SYSTEM, T2_USER, label="T2: Guided CoT")
print()
display_result(result_t2)

[T2: Guided CoT] Sending request...
[T2: Guided CoT] Status  : OK
[T2: Guided CoT] Tokens  : 547 in / 137 out

  T2: Guided CoT
  Incident Type  : Phishing
  Severity       : HIGH
  Confidence     : 0.900
  Summary        : Multiple employees in the Finance department received phishing emails that led
them to click on links and enter their credentials on fake Office 365 login pages. This resulted in
immediate disabling of the employee's account.
  Actions (3):
    1. Notify affected users about the phishing attempt and educate them on how to recognize such attacks.
    2. Review and update security policies related to email authentication and user training.
    3. Improve monitoring tools to detect and respond more quickly to similar phishing attempts.
  Tokens         : 547 in / 137 out


---
## 7. Técnica 3 — Meta Prompting

### Racional de desenho

**Meta prompting** usa o próprio LLM para gerar ou melhorar um prompt e depois aplica esse prompt à tarefa real. Essa abordagem em duas fases pode revelar estruturas de prompt que um engenheiro humano talvez não desenhasse diretamente.

**Fase 1 — Geração do prompt:** pedir ao mesmo modelo open source que desenhe o melhor prompt possível para análise de incidentes.

**Fase 2 — Aplicação do prompt:** usar o prompt gerado (mais o texto do incidente) para a análise efetiva.

### Fluxo de iteração

```
[v1: naive] → [meta: generate optimal prompt] → [v2: generated] → [run analysis]
```

### Comportamento esperado
O prompt gerado pode incorporar escolhas estruturais inesperadas, enquadramentos novos e restrições adicionais, potencialmente melhorando a qualidade final.

In [40]:
META_GEN_SYSTEM = (
    "You are an expert prompt engineer specializing in designing AI prompts for high-stakes "
    "security operations systems. You understand how to elicit accurate, calibrated, and "
    "structured outputs from large language models."
)

META_INCIDENT_PLACEHOLDER = "{{INCIDENT_REPORT}}"

_SCHEMA_SPEC = (
    '{ "incident_type": str, "severity": "low|medium|high|critical", '
    '"confidence": float 0-1, "summary": str, "recommended_actions": list[str] }'
)

def build_structured_meta_prompt(extra_guidance=""):
    guidance_block = ""
    if extra_guidance:
        guidance_block = (
            "\n\n## Additional Guidance\n"
            + extra_guidance
        )

    return (
        "## Role\n"
        "You are a senior cybersecurity analyst responsible for SOC triage, incident classification, "
        "and immediate response recommendations.\n\n"
        "## Objective\n"
        "Analyze the incident report and produce a precise, evidence-based assessment.\n\n"
        "## Analysis Workflow\n"
        "1. Identify the initial attack vector and key indicators.\n"
        "2. Choose the single most likely incident type.\n"
        "3. Assess severity using business impact, scope, urgency, and CIA impact.\n"
        "4. Estimate confidence based only on evidence present in the report.\n"
        "5. Write a concise factual summary in 2-3 sentences.\n"
        "6. List prioritized SOC response actions.\n\n"
        "## Decision Rules\n"
        "- Base conclusions only on evidence from the incident report.\n"
        "- Do not invent tools, attackers, or impacts not supported by the report.\n"
        "- Choose exactly one incident_type.\n"
        "- severity must be exactly one of: low, medium, high, critical.\n"
        "- confidence must be a float between 0.0 and 1.0.\n"
        "- If evidence is incomplete, lower confidence instead of speculating.\n"
        "- recommended_actions must be specific, actionable, and ordered by priority."
        + guidance_block
        + "\n\n## Incident Report\n"
        + META_INCIDENT_PLACEHOLDER
        + "\n\n## Output Requirements\n"
        "Respond with ONLY a valid JSON object. Do not include markdown fences, headings, bullet points, "
        "or explanatory text.\n"
        "Use exactly this schema:\n"
        + _FINAL_JSON
    )


def normalize_meta_prompt(raw_prompt):
    cleaned = raw_prompt.strip()
    cleaned = re.sub(r"^```(?:\w+)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    required_sections = [
        "## Role",
        "## Objective",
        "## Analysis Workflow",
        "## Decision Rules",
        "## Incident Report",
        "## Output Requirements",
    ]
    has_required_structure = all(section in cleaned for section in required_sections)
    has_placeholder = META_INCIDENT_PLACEHOLDER in cleaned
    enforces_json_only = "ONLY a valid JSON object" in cleaned or "Return ONLY JSON" in cleaned

    if has_required_structure and has_placeholder and enforces_json_only:
        return cleaned

    compressed_guidance = " ".join(cleaned.split())
    return build_structured_meta_prompt(extra_guidance=compressed_guidance[:500])


META_GEN_USER = (
    "Design a production-ready prompt for a cybersecurity incident analysis assistant.\n\n"
    "Return a complete prompt, not an answer to the incident.\n"
    "The prompt you write must contain exactly these section headers:\n"
    "## Role\n"
    "## Objective\n"
    "## Analysis Workflow\n"
    "## Decision Rules\n"
    "## Incident Report\n"
    "## Output Requirements\n\n"
    "Requirements for the generated prompt:\n"
    "1. It must classify diverse cybersecurity incidents accurately.\n"
    "2. It must calibrate severity as low/medium/high/critical using evidence.\n"
    "3. It must assign a confidence float from 0.0 to 1.0.\n"
    "4. It must produce a concise factual summary.\n"
    "5. It must produce prioritized SOC response actions.\n"
    "6. It must enforce strict JSON output using this schema:\n"
    "   " + _SCHEMA_SPEC + "\n"
    "7. In the Incident Report section, use the literal placeholder "
    + META_INCIDENT_PLACEHOLDER
    + " exactly once.\n"
    "8. Do not include sample answers, example JSON instances, or markdown fences.\n"
    "9. Do not return commentary about prompt engineering. Return only the final prompt text.\n\n"
    "The prompt must be robust across: Ransomware, Phishing, DDoS, Insider Threat, Credential Theft, "
    "Malware, and Supply Chain attacks."
)

print("Phase 1: Generating optimal analysis prompt via meta-prompting...")
print()

RAW_GENERATED_PROMPT, meta_input_tokens, meta_output_tokens = generate_response_text(
    META_GEN_SYSTEM,
    META_GEN_USER,
    max_new_tokens=512,
)
GENERATED_PROMPT = normalize_meta_prompt(RAW_GENERATED_PROMPT)

print("Meta-prompt generated.")
print(f"Tokens: {meta_input_tokens} in / {meta_output_tokens} out")
print(f"Normalized: {'YES' if GENERATED_PROMPT != RAW_GENERATED_PROMPT else 'NO'}")
print()
print("─" * 65)
print("GENERATED PROMPT:")
print("─" * 65)
print_wrapped(GENERATED_PROMPT)

Phase 1: Generating optimal analysis prompt via meta-prompting...



Meta-prompt generated.
Tokens: 319 in / 89 out
Normalized: YES

─────────────────────────────────────────────────────────────────
GENERATED PROMPT:
─────────────────────────────────────────────────────────────────
## Role
You are a senior cybersecurity analyst responsible for SOC triage, incident classification, and
immediate response recommendations.

## Objective
Analyze the incident report and produce a precise, evidence-based assessment.

## Analysis Workflow
1. Identify the initial attack vector and key indicators.
2. Choose the single most likely incident type.
3. Assess severity using business impact, scope, urgency, and CIA impact.
4. Estimate confidence based only on evidence present in the report.
5. Write a concise factual summary in 2-3 sentences.
6. List prioritized SOC response actions.

## Decision Rules
- Base conclusions only on evidence from the incident report.
- Do not invent tools, attackers, or impacts not supported by the report.
- Choose exactly one incident_typ

In [41]:
# Comentário em português: a análise final continua usando prompts em inglês.
T3_SYSTEM = (
    "You are a senior cybersecurity analyst. Follow the user's instructions exactly and return "
    "only a valid JSON object. Never use markdown fences, headings, or explanatory prose."
)

T3_USER = GENERATED_PROMPT.replace(META_INCIDENT_PLACEHOLDER, INCIDENT_TEXT)
if META_INCIDENT_PLACEHOLDER in T3_USER:
    raise ValueError("Meta prompt still contains the incident placeholder after injection.")

print("Phase 2: Running analysis with the generated prompt...")
print()
result_t3 = run_analysis(T3_SYSTEM, T3_USER, label="T3: Meta Prompting")
print()
display_result(result_t3)

Phase 2: Running analysis with the generated prompt...

[T3: Meta Prompting] Sending request...


[T3: Meta Prompting] Status  : OK
[T3: Meta Prompting] Tokens  : 538 in / 122 out

  T3: Meta Prompting
  Incident Type  : Phishing
  Severity       : HIGH
  Confidence     : 0.900
  Summary        : Multiple employees received phishing emails demanding urgent wire transfers to a
fake Office 365 login page. One employee fell victim and had their account disabled.
  Actions (3):
    1. Notify all affected users about the phishing attempt and educate them on how to recognize such scams.
    2. Review and update internal security policies regarding phishing threats.
    3. Implement additional monitoring and alerts for suspicious activity related to financial transactions.
  Tokens         : 538 in / 122 out


---
## 8. Demonstração de Iteração de Prompts

As três técnicas representam uma evolução desde um prompt ingênuo até desenhos progressivamente mais sofisticados.

```
v1 (Base) ──► v2 (RCTOF) ──► v3 (Guided CoT) ──► v4 (Meta-gerado)
    ↓              ↓                 ↓                    ↓
Sem estrutura  Papel explícito   Raciocínio guiado   Estrutura desenhada pelo LLM
Sem schema     + schema JSON     passo a passo       automaticamente
```

In [42]:
V1_PROMPT = (
    "Analyze this cybersecurity incident and output a JSON object with: "
    "incident_type, severity, confidence, summary, recommended_actions.\n\n"
    + INCIDENT_TEXT
)

prompts = [
    ("v1 — Baseline (no structure)", V1_PROMPT),
    ("v2 — RCTOF (Technique 1)", T1_USER),
    ("v3 — Guided CoT (Technique 2)", T2_USER),
    ("v4 — Meta-generated (Technique 3)", T3_USER),
]

for label, prompt in prompts:
    print(f"{'─' * 65}")
    print(f"  {label}")
    print(f"  Length: {len(prompt)} chars")
    print()
    preview = " ".join(prompt[:280].split())
    print_wrapped(f"  Preview: {preview}...")
    print()

─────────────────────────────────────────────────────────────────
  v1 — Baseline (no structure)
  Length: 689 chars

  Preview: Analyze this cybersecurity incident and output a JSON object with: incident_type,
severity, confidence, summary, recommended_actions. On June 14th, multiple employees in the Finance
department reported receiving an email claiming to be from the CFO, requesting urgent wire transf...

─────────────────────────────────────────────────────────────────
  v2 — RCTOF (Technique 1)
  Length: 1327 chars

  Preview: ## Context Your organization's SOC has received a cybersecurity incident report requiring
immediate triage. ## Task Analyze the incident report below. Identify the attack type, assess
severity, estimate your confidence level, write a concise summary, and list prioritized respons...

─────────────────────────────────────────────────────────────────
  v3 — Guided CoT (Technique 2)
  Length: 2190 chars

  Preview: Analyze this cybersecurity incident using the 

---
## 9. Tabela Comparativa

Resultados lado a lado das três técnicas sobre o mesmo incidente.

In [43]:
all_results = [result_t1, result_t2, result_t3]

rows = []
for r in all_results:
    analysis = r.get("analysis")
    rows.append({
        "Technique": r["label"],
        "Parse OK": "YES" if r["parse_ok"] else "NO",
        "Valid": "YES" if r["validation_ok"] else "NO",
        "Incident Type": analysis.incident_type if analysis else "—",
        "Severity": analysis.severity.value if analysis else "—",
        "Confidence": f"{analysis.confidence:.3f}" if analysis else "—",
        "# Actions": len(analysis.recommended_actions) if analysis else 0,
        "In Tok": r["input_tokens"],
        "Out Tok": r["output_tokens"],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()

# Compara a consistência geral apenas quando todas as saídas passam na validação.
if all(r["validation_ok"] for r in all_results):
    types = {r["analysis"].incident_type for r in all_results}
    severities = {r["analysis"].severity.value for r in all_results}
    confidences = [r["analysis"].confidence for r in all_results]

    print(f"Incident type agreement : {types}")
    print(f"Severity agreement      : {severities}")
    print(f"Confidence range        : {min(confidences):.3f} – {max(confidences):.3f}")
    print(f"Confidence spread       : {max(confidences) - min(confidences):.3f}")

         Technique Parse OK Valid Incident Type Severity Confidence  # Actions  In Tok  Out Tok
         T1: RCTOF      YES   YES      Phishing     high      0.900          2     327       94
    T2: Guided CoT      YES   YES      Phishing     high      0.900          3     547      137
T3: Meta Prompting      YES   YES      Phishing     high      0.900          3     538      122

Incident type agreement : {'Phishing'}
Severity agreement      : {'high'}
Confidence range        : 0.900 – 0.900
Confidence spread       : 0.000


---
## 10. Discussão

### Técnica 1 — RCTOF

**Pontos fortes:**
- Overhead mínimo de tokens, com uma única geração e prompt curto
- Estrutura previsível de saída, pois o esquema é imposto no próprio prompt
- Boa aderência a pipelines de classificação em alto volume e sensíveis a latência

**Limitações:**
- Não oferece trilha explícita de raciocínio para auditoria
- Pode falhar silenciosamente em incidentes ambíguos, com confiança inflada
- Aderência ao schema depende do wording do prompt e do comportamento do modelo

---

### Técnica 2 — Guided Chain of Thought

**Pontos fortes:**
- Tendência a melhor calibração de confiança, já que o prompt força avaliação baseada em evidências
- Melhores notas de severidade, pois a tríade CIA fornece uma base mais disciplinada
- Lida melhor com incidentes ambíguos, fazendo a incerteza aparecer com mais naturalidade

**Limitações:**
- Custo maior em tokens por causa do prompt mais longo
- Latência maior em comparação com a técnica RCTOF
- Pode ser excessiva para incidentes muito claros e diretos

---

### Técnica 3 — Meta Prompting

**Pontos fortes:**
- Revela estruturas de prompt que o engenheiro talvez não tivesse considerado
- Útil como ferramenta de descoberta para novas categorias de incidente ou novos domínios
- Permite reexecutar a etapa meta quando os tipos de incidente ou o contexto organizacional mudarem

**Limitações:**
- A geração do prompt é não determinística e pode variar entre execuções
- Exige duas gerações, aumentando custo computacional e tempo total
- O prompt gerado ainda precisa de revisão humana antes de ir para produção
- Pode introduzir restrições alucinadas ou enquadramentos subótimos

---

### Validação de Schema com Pydantic

O uso de Pydantic fornece:
- **Mensagens de erro por campo** para depurar respostas malformadas
- **Enforcement de Enum** em `severity`, bloqueando strings inválidas antes de propagarem
- **Validação de intervalo** em `confidence`, evitando floats fora da faixa esperada
- Um contrato de schema reutilizável e versionável, independente da técnica de prompting

A cadeia de fallback em `parse_json_response` (direto → remoção de fences → regex gulosa) cobre os modos de falha mais comuns sem sacrificar precisão.

---
## 11. Conclusão

Este notebook demonstrou três técnicas de prompt engineering aplicadas à triagem de incidentes de cibersegurança com um **modelo open source do Hugging Face**:

| Técnica | Melhor encaixe | Compromisso principal |
|---------|----------------|-----------------------|
| **RCTOF** | Pipelines de alto volume, incidentes claros | Rápida e simples, mas sem trilha de raciocínio |
| **Guided CoT** | Incidentes ambíguos, decisões sensíveis à calibração | Mais consistente, porém com prompts mais longos |
| **Meta Prompting** | Descoberta de prompts, novas categorias | Estrutura nova, porém com variabilidade entre execuções |

### Principais conclusões

1. **A estrutura do prompt importa mais do que o tamanho.** Um schema bem especificado reduz a maior parte das falhas de parsing de JSON, independentemente da técnica.

2. **O raciocínio guiado tende a melhorar a calibração.** As notas de confiança e severidade ficam mais ancoradas nas evidências quando o prompt força uma passagem disciplinada pela tríade CIA.

3. **A validação com Pydantic é uma barreira rígida.** Sistemas SOC a jusante não devem processar saídas de LLM sem validação prévia; o schema é o contrato entre o modelo e o pipeline.

4. **Meta prompting é ferramenta de pesquisa, não atalho de produção.** Prompts gerados precisam de revisão humana e versionamento como qualquer outro artefato de engenharia.